# Designing the lens system to collimate a beam after an optical fiber

The Optical fiber information we have is:
- Customer: AMS Technologies AG
- Part QPMJ-A3A, 3S-400-3/125-3-2-1
- Fiber Core/Cladding/Jacket Diameter (in micron): 3/125/3000
- Numerical Aperture (NA): 0.11
- Length (m): 1x (2 $\pm$ 0.1) and 3x (5 $\pm$ 0.1)
- Fiber Type: QP, Single Mode, Polarization Maintaining
- Wavelength (nm): 405

Using solely the NA to predict the beam propagation can lead to great inaccuracies, as seen at the (Thorlabs Insights)[https://www.thorlabs.com/numerical-aperture-is-not-a-good-parameter-for-single-mode-fibers?tabName=Insights].

A better option is to use the mode field diameter (MFD) as the Gaussion beam diameter at the waist.
Therefore, we have to compute the MFD which can be approximated for a single-mode fiber as,
$$MFD = \frac{2 \lambda}{\pi \cdot NA}$$
following [Schafter+Kirchhoff MFD computation](https://www.sukhamburg.com/support/technotes/fiberoptics/cablebasics/mfd.html), thus considering the MFD and NA for a 1/$e^2$ of a Gaussian beam.

## Gaussian Beam Equations
The Gaussian beam propagation can be described using the following equations:
- Beam waist ($w_0$):
$$w_0 = \frac{MFD}{2}$$
- Far-field divergence angle ($\theta$):
$$\theta = \frac{\lambda}{\pi \cdot w_0}$$
- Beam radius at distance $z$ from the waist ($w(z)$):
$$w(z) = w_0 \sqrt{1 + \left(\frac{z}{z_R}\right)^2}$$
where $z_R$ is the Rayleigh range, given by:
$$z_R = \frac{\pi w_0^2}{\lambda}$$

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, FloatText
%matplotlib inline

In [19]:
def simulate_beam_propagation(wavelength_nm=405.0, NA=0.11, target_radius_mm=6.5, z_max_mm=100.0):
    """
    Simulate Gaussian beam propagation and find z for target radius.
    All calculations in mm.
    """
    # Convert wavelength to mm
    wavelength = wavelength_nm / 1e6
    
    # Calculate MFD in mm
    MFD = (2 * wavelength) / (np.pi * NA)
    
    # Calculate beam waist w0 in mm
    w0 = MFD / 2
    
    # Calculate Rayleigh range in mm
    z_R = (np.pi * w0**2) / wavelength
    
    # Calculate far-field divergence angle in radians
    theta = wavelength / (np.pi * w0)
    
    # Create z array in mm
    z = np.linspace(0, z_max_mm, 1000)
    
    # Calculate beam radius at each z in mm
    w_z = w0 * np.sqrt(1 + (z / z_R)**2)
    
    # Calculate z for target radius
    if target_radius_mm > w0:
        z_target = z_R * np.sqrt((target_radius_mm / w0)**2 - 1)
    else:
        z_target = 0.0
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot w(z)
    ax1.plot(z, w_z, 'b-', linewidth=2)
    ax1.axhline(w0, color='r', linestyle='--', label=f'Beam waist $w_0$ = {w0*1e3:.3f} \u03bcm')
    ax1.axvline(z_R, color='g', linestyle='--', label=f'Rayleigh range $z_R$ = {z_R:.3f} mm')
    ax1.axhline(target_radius_mm, color='k', linestyle=':', label=f'Target radius = {target_radius_mm:.3f} mm')
    ax1.axvline(z_target, color='k', linestyle=':', label=f'z for target = {z_target:.3f} mm')
    ax1.set_xlabel('Distance z (mm)')
    ax1.set_ylabel('Beam radius w(z) (mm)')
    ax1.set_title('Gaussian Beam Propagation')
    ax1.legend()
    ax1.grid(True)
    
    # Plot w(z) in log scale for better visualization of divergence
    ax2.plot(z, w_z, 'b-', linewidth=2)
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_xlabel('Distance z (mm) - log scale')
    ax2.set_ylabel('Beam radius w(z) (mm) - log scale')
    ax2.set_title('Gaussian Beam Propagation (log-log)')
    ax2.grid(True, which='both')
    
    plt.tight_layout()
    plt.show()
    
    # Print parameters
    print(f'Wavelength: {wavelength_nm} nm ({wavelength*1e3:.6f} \u03bcm)')
    print(f'Numerical Aperture: {NA}')
    print(f'MFD: {MFD*1e3:.3f} \u03bcm')
    print(f'Beam waist w0: {w0*1e3:.3f} \u03bcm')
    print(f'Rayleigh range z_R: {z_R:.3f} mm')
    print(f'Far-field divergence angle: {np.degrees(theta):.6f} degrees ({theta*1e3:.6f} mrad)')
    print(f'\nDistance z to achieve radius w = {target_radius_mm:.3f} mm:')
    print(f'  z = {z_target:.6f} mm')
    
    return z_target

In [20]:
# Create interactive sliders
interact(
    simulate_beam_propagation,
    wavelength_nm=FloatSlider(min=390, max=450, step=1, value=423, description='Wavelength (nm):'),
    NA=FloatSlider(min=0.01, max=0.3, step=0.01, value=0.11, description='NA:'),
    target_radius_mm=FloatSlider(min=4, max=20, step=0.1, value=6.5, description='Target radius (mm):'),
    z_max_mm=FloatSlider(min=10, max=200, step=10, value=100, description='Max z (mm):')
)

interactive(children=(FloatSlider(value=423.0, description='Wavelength (nm):', max=450.0, min=390.0, step=1.0)…

<function __main__.simulate_beam_propagation(wavelength_nm=405.0, NA=0.11, target_radius_mm=6.5, z_max_mm=100.0)>

## Explanation

The interactive simulation above allows you to:
- Adjust the **wavelength** (default: 405 nm from your fiber spec)
- Adjust the **Numerical Aperture** (default: 0.11 from your fiber spec)
- Set a **target beam radius** (default: 6.5 mm)
- Set the **maximum z range** for the plot (default: 100 mm)

The left plot shows the beam radius as a function of distance from the waist in linear scale.
The right plot shows the same in log-log scale, which is useful for visualizing the far-field divergence.

The calculation prints:
- The computed MFD (Mode Field Diameter)
- The beam waist w0 = MFD/2
- The Rayleigh range z_R
- The far-field divergence angle
- The distance z required to achieve the target beam radius

**Note:** This notebook uses `%matplotlib inline` for compatibility. For interactive plots that update with the sliders, install `ipympl`:
```
pip install ipympl
```
Then change `%matplotlib inline` to `%matplotlib widget` in the imports cell and restart the kernel.

All printed values are in mm (or micrometers for wavelength), and the target radius calculation uses the formula:
$$z = z_R \sqrt{\left(\frac{w}{w_0}\right)^2 - 1}$$